# Quick Start · EasySewer 2.0 候选 API

本例保留原来的 cubic 网络：读取模型、查看对象和地图、切换雨事件、导出、求解及读取六条结果序列。
当前开发版本为 `2.0.0.dev0`，使用 `easysewer.model.Model`；顶层 `easysewer.Model` 仍是旧接口。

在安装了当前候选 EasySewer 包的 Python 3.10+ 内核中执行。绘图需要 Plotly（本例验证版本 6.0.1）；
从仓库根目录或本笔记本目录启动均可。求解需要 Windows/Linux 原生库。
纯 Python 包可执行到“导出模型”，也可独立读取已有结果存档；不能执行求解。
安装包与发布状态见仓库 README，本例不代表 2.0 已发布。

In [ ]:
from pathlib import Path
from datetime import timedelta
import sys
import tempfile

from easysewer import Model
from easysewer.model import Ref, FileReference
from easysewer.model.hydrology import SeriesRainfall
from easysewer.model.resources import InlineTimeSeries, FileTimeSeries
from easysewer.io.json import JsonDocument
from easysewer.runtime import RunConfig, RunResult

cwd = Path.cwd().resolve()
repo = next((p for p in (cwd, *cwd.parents)
             if (p / 'cases/Case_1 Quick Start/cubic.inp').is_file()), None)
if repo is None:
    raise FileNotFoundError('请从 EasySewer 仓库或本笔记本目录启动内核')
sys.path.insert(0, str(repo))  # 仅用于仓库中的绘图辅助模块。
from cases.utils import plot_model

input_path = repo / 'cases/Case_1 Quick Start/cubic.inp'
backend = 'swmm:standard'

## 1. 读取与可视化

`strict=True` 会报告模型校验错误。集合按对象 ID 访问，通过 `.values()` 遍历对象。
模型包含 14 个节点（13 个检查井和 1 个出口）、13 条管道和 8 个汇水区。
地图保留输入坐标单位；水力长度和高程单位另行显示。

In [ ]:
cubic = Model.from_inp(input_path, strict=True)
print('nodes:', len(cubic.nodes), 'links:', len(cubic.links),
      'subcatchments:', len(cubic.subcatchments))
fig = plot_model(cubic)
fig.show()

## 2. 查看设置及对象

`options` 保存输入中声明的值，省略项为 `None`；`effective_options` 给出包含缺省值的解析设置。
时间使用 Python 的日期、时间和 `timedelta`，模型日历不附加时区。
本例流量单位 LPS，长度单位 m，汇水区面积单位 ha；不能把地图坐标默认当成米。

In [ ]:
settings = cubic.effective_options.values
for name in ('flow_units', 'infiltration', 'flow_routing', 'start_date',
             'start_time', 'end_date', 'end_time', 'routing_step',
             'report_start_date', 'report_start_time', 'report_step'):
    print(name, getattr(settings, name))

for link in cubic.links.values():
    print(link.id, link.length, cubic.units.unit('length'), link.inlet, link.outlet)
for node in cubic.nodes.values():
    print(node.id, node.position, node.elevation, cubic.units.unit('elevation'))
for area in cubic.subcatchments.values():
    print(area.id, area.area, cubic.units.unit('catchment_area'), area.outlet, area.rain_gage)

## 3. 查看降雨并切换事件

`RainSeries-1` 和 `RainSeries-2` 是此输入自带的两场雨。本例两者均按同一雨量站的
INTENSITY 格式和一分钟间隔解释。更换任意外部雨事件前，应先核对格式、间隔和单位。
对象为不可变记录，通过集合的 `update` 修改；`Ref` 同时记录目标集合和 ID。

In [ ]:
for gage in cubic.raingages.values():
    print(gage.id, gage.form, gage.interval, gage.snow_factor, gage.source)
for series in cubic.timeseries.values():
    if isinstance(series, InlineTimeSeries):
        print(series.id, 'inline', len(series.points), series.points[:5])
    elif isinstance(series, FileTimeSeries):
        print(series.id, 'external file', series.file)

cubic.raingages.update('28_RG1', source=SeriesRainfall(
    series=Ref(collection='swmm:timeseries', key='RainSeries-2')))
cubic.validate(for_run=True).raise_for_errors()
print('Selected:', cubic.raingages['28_RG1'].source)

## 4. 导出模型

每次执行创建独立输出目录，保留原始 `cubic.inp` 和以前的结果。INP 可用于 SWMM；JSON
保存 2.0 模型及其来源信息。这里重新读取两种文件，核对换雨修改和主要集合。

In [ ]:
output_root = repo / 'runs/quick-start'
output_root.mkdir(parents=True, exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix='cubic-', dir=output_root))
cubic.to_inp(output_dir / 'cubic_r2.inp')
cubic.to_json_document().write(output_dir / 'cubic_r2.json')
from_inp = Model.from_inp(output_dir / 'cubic_r2.inp', strict=True)
from_json = Model.from_json_document(JsonDocument.read(output_dir / 'cubic_r2.json'), strict=True)
for restored in (from_inp, from_json):
    for collection in ('nodes', 'links', 'subcatchments', 'raingages', 'timeseries'):
        assert tuple(getattr(restored, collection).values()) == tuple(getattr(cubic, collection).values())
fig.write_html(output_dir / 'network.html', include_plotlyjs=True)
print('Output directory:', output_dir)

## 5. 求解与存档（需要原生库）

`run` 在独立求解进程中运行模型快照。`raise_for_status()` 将失败明确抛出，避免继续读取不存在的结果。
调用超时和总时限分别设置；结果存档包含运行信息和输出文件，可以移动后再打开。
本例使用标准 SWMM 后端。

In [ ]:
result = cubic.run(RunConfig(
    backend=backend,
    output_directory=FileReference(path=str(output_dir / 'run'), direction='output'),
    artifact_stem='cubic_r2',
    native_call_timeout=timedelta(seconds=30),
    wall_time_limit=timedelta(minutes=2),
))
result.raise_for_status()
archive_dir = output_dir / 'archive'
result.save(archive_dir)
print('Status:', result.status, 'archive:', archive_dir)

## 6. 按对象 ID 读取结果

下面只需 Python 结果读取器。新内核可先导入 `RunResult`、`Ref` 和 `Path`，再把 `archive_dir`
设为已有存档目录，不需要重新求解。

使用 OUT 中记录的对象 ID，不依赖可编辑模型的列表下标。原例的节点数值通道 4、0、5
分别对应 `swmm:inflow`、`swmm:depth`、`swmm:overflow`；管道通道 0、1、2
对应 `swmm:flow`、`swmm:depth`、`swmm:velocity`。
`overflow` 的具体含义保留在序列 `semantics` 中，不能无条件解释为所有后端的累计洪水量。
时间直接返回 `datetime`，单位、采样方式和来源与数据一起保留。

In [ ]:
archived = RunResult.load(archive_dir)
archived.raise_for_status()
queries = (
    ('node_inflow', Ref(collection='swmm:nodes', key='N1'), 'swmm:inflow'),
    ('node_depth', Ref(collection='swmm:nodes', key='N1'), 'swmm:depth'),
    ('node_overflow', Ref(collection='swmm:nodes', key='N1'), 'swmm:overflow'),
    ('link_flow', Ref(collection='swmm:links', key='L1'), 'swmm:flow'),
    ('link_depth', Ref(collection='swmm:links', key='L1'), 'swmm:depth'),
    ('link_velocity', Ref(collection='swmm:links', key='L1'), 'swmm:velocity'),
)
with archived.open_output() as output:
    print('Recorded node IDs:', output.metadata.names('swmm:nodes'))
    print('Recorded link IDs:', output.metadata.names('swmm:links'))
    print('Saved periods:', output.metadata.periods,
          'from:', output.metadata.first_time, 'to:', output.metadata.last_time)
    results = {label: output.series(target, variable) for label, target, variable in queries}

for label, series in results.items():
    print(label, series.unit, series.sampling, series.semantics)
    print(list(zip(series.times[:5], series.values[:5])))
    peak = series.maximum()
    print('Maximum among saved samples:', peak.value, peak.unit, peak.time)
    series.to_json_document().write(output_dir / (label + '.json'))

以上最大值只比较已保存的报告时刻，不能代替求解器内部步长的峰值。
原例的 `node.index_of(...)` / `link.index_of(...)` 查询在此被有集合类型的 `Ref` 替代。
模型编辑不会改变已保存结果的对象身份。

进一步阅读：[模型 API](../../docs/2.0-model.md)、[结果读取](../../docs/2.0-output.md)、
[1.x 配置 JSON 迁移](../../docs/2.0-migrate-config.md)。